In [1]:
import httpx
from bs4 import BeautifulSoup
import asyncio
import random
from user_agents import get_random_user_agent, get_random_header

In [2]:
BASE_URL = 'https://www.vademecum.es'
QUERY_URL = '/espana/es/alfa'
URL = BASE_URL + QUERY_URL
OUTPUT_FOLDER = 'corpus/vademecum_es/'

In [ ]:
async def get_html(url):
    async with httpx.AsyncClient() as client:
        headers = get_random_header()
        try:
            await asyncio.sleep(random.random() + 1) 
            response = await client.get(url, follow_redirects=True, headers = headers, timeout=60.0)
            if response.status_code == 200:
                return response.text
            else:
                return response
        except Exception as e:
            print(repr(e), url)
            return None

async def writeMedData(med_url, filename):
    med_page = await get_html(med_url)
    if med_page is not None:
        med_soup = BeautifulSoup(med_page, 'html.parser')
        med_html = med_soup.find("div", class_="bodytext3_mono")
        if med_html is None: med_html = med_soup.find("div", id="fichaATC")
        if med_html is not None:
            med_txt = med_html.getText()
            lines = [line.strip() for line in med_txt.split('\n') if line.strip()]
            f = open(OUTPUT_FOLDER + filename, "w")
            for l in lines[:-2]: 
                f.write(l + "\n")
            f.close()
    

In [ ]:
with open("vademecum_es.txt") as tf:
    n = 0
    for line in tf:
        url = line.split(" : ")[1].strip()
        fname = f'VAD{n:05}.txt'
        await writeMedData(url, fname)
        print(fname, url)
        n += 1
        

In [ ]:
import os

PATH = "corpus/vademecum_es/"

n = 0
for cont in range(9219):
    fname = f'VAD{cont:05}.txt'
    if not os.path.exists(PATH + fname):
        print(fname)
        n += 1
print("Total:", n)

VAD00612.txt
VAD00758.txt
VAD01462.txt
VAD01963.txt
VAD02249.txt
VAD02344.txt
VAD02454.txt
VAD03050.txt
VAD03143.txt
VAD03181.txt
VAD03341.txt
VAD03428.txt
VAD04044.txt
VAD04164.txt
VAD04202.txt
VAD04231.txt
VAD04313.txt
VAD04315.txt
VAD04421.txt
VAD04922.txt
VAD05039.txt
VAD05048.txt
VAD05113.txt
VAD05287.txt
VAD05724.txt
VAD06071.txt
VAD06410.txt
VAD06559.txt
VAD06794.txt
VAD06879.txt
VAD07011.txt
VAD07265.txt
VAD07410.txt
VAD08320.txt
VAD08665.txt
Total: 35


In [ ]:
import os
import hashlib

def calculate_file_hash(file_path):
    """Calculates the hash value of a file's content."""
    hasher = hashlib.md5()
    with open(file_path, 'rb') as file:
        for chunk in iter(lambda: file.read(4096), b''):
            hasher.update(chunk)
    return hasher.hexdigest()

def find_duplicate_files(root_folder):
    """Traverses through the root folder and identifies duplicate files."""
    duplicates = {}
    for folder_path, _, file_names in os.walk(root_folder):
        for file_name in file_names:
            file_path = os.path.join(folder_path, file_name)
            file_hash = calculate_file_hash(file_path)
            if file_hash in duplicates:
                duplicates[file_hash].append(file_path)
            else:
                duplicates[file_hash] = [file_path]
    return duplicates

def remove_duplicate_files(duplicates):
    """Removes duplicate files from the file system."""
    for file_paths in duplicates.values():
        if len(file_paths) > 1:
            print(f"Duplicate files found:\n{file_paths}\n")
            for file_path in file_paths[1:]:
                os.remove(file_path)
                print(f"{file_path} has been deleted.\n")


root_folder = 'corpus/vademecum_es/'
duplicates = find_duplicate_files(root_folder)
remove_duplicate_files(duplicates)


Duplicate files found:
['corpus/vademecum_es/VAD00694.txt', 'corpus/vademecum_es/VAD04538.txt', 'corpus/vademecum_es/VAD04532.txt', 'corpus/vademecum_es/VAD04539.txt', 'corpus/vademecum_es/VAD04531.txt', 'corpus/vademecum_es/VAD04530.txt', 'corpus/vademecum_es/VAD04535.txt', 'corpus/vademecum_es/VAD04536.txt', 'corpus/vademecum_es/VAD00698.txt', 'corpus/vademecum_es/VAD00693.txt', 'corpus/vademecum_es/VAD00692.txt', 'corpus/vademecum_es/VAD04529.txt', 'corpus/vademecum_es/VAD00691.txt', 'corpus/vademecum_es/VAD00696.txt', 'corpus/vademecum_es/VAD04534.txt', 'corpus/vademecum_es/VAD00697.txt', 'corpus/vademecum_es/VAD00695.txt', 'corpus/vademecum_es/VAD04533.txt', 'corpus/vademecum_es/VAD04540.txt']

corpus/vademecum_es/VAD04538.txt has been deleted.

corpus/vademecum_es/VAD04532.txt has been deleted.

corpus/vademecum_es/VAD04539.txt has been deleted.

corpus/vademecum_es/VAD04531.txt has been deleted.

corpus/vademecum_es/VAD04530.txt has been deleted.

corpus/vademecum_es/VAD04535.tx